# Ranking Evaluation of Trained Models

This notebook loads previously trained models (MLP, LightGBM, XGBoost, GNN+MLP) and evaluates them on the ranking task.
Computed metrics: `roc_auc`, `mrr`, `ndcg_at_5`, `ndcg_at_10`, `ndcg_at_50`, `ndcg_at_100`.

In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

'1'

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR   = (PROJECT_ROOT / 'data' / 'link_prediction' / 'llm_concept_datasets').resolve()
OUTPUT_DIR = (PROJECT_ROOT / 'data' / 'link_prediction' / 'llm_concept_results').resolve()
MODEL_DIR  = OUTPUT_DIR / 'models'

RANKING_CSV = DATA_DIR / 'ranking_test.csv'
TRAIN_CSV   = DATA_DIR / 'train.csv'
VAL_CSV     = DATA_DIR / 'val.csv'

# Graph file used to train the GNN (TRAIN-window graph to avoid leakage).
# Adjust the filename if your graph file is named differently.
GRAPH_FILE = DATA_DIR / 'graph_2017_2022.pkl'

K_VALUES = [5, 10, 50, 100]

assert RANKING_CSV.is_file(), f'Missing {RANKING_CSV}'
assert TRAIN_CSV.is_file(),   f'Missing {TRAIN_CSV}'
assert MODEL_DIR.is_dir(),    f'Missing {MODEL_DIR}'

print('Checkpoints found:')
for p in sorted(MODEL_DIR.iterdir()):
    print(' ', p.name)

print('\nGraph file:', GRAPH_FILE, '(exists =', GRAPH_FILE.is_file(), ')')

Checkpoints found:
  gnn_mlp_gnn_emb.pt
  gnn_mlp_gnn_struct_emb.pt
  lightgbm_all.pkl
  mlp_all.pt
  mlp_emb.pt
  mlp_structure.pt
  xgboost_all.pkl

Graph file: /Users/rufina2304/Desktop/project_mfdp/Concept2Paper/data/link_prediction/llm_concept_datasets/graph_2017_2022.pkl (exists = True )


In [4]:
import json
import numpy as np
import pandas as pd

from src.link_prediction.calculate_metrics import (
    compute_ranking_metrics,
    format_ranking_metrics,
)
from src.link_prediction.load_data import (
    classify_features,
    _select_cols,
    load_datasets,
    GNNData,
)
from src.link_prediction.models.graph_models import HAS_TORCH_GEOMETRIC

print('torch_geometric available:', HAS_TORCH_GEOMETRIC)

/Users/rufina2304/Desktop/project_mfdp/Concept2Paper/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch_geometric available: True


## Model registry & scalers

In [5]:
# Map: model display name -> (checkpoint stem, kind, feature_groups)
MODEL_REGISTRY = [
    ('MLP (structure)', 'mlp_structure', 'mlp',  'structure'),
    ('MLP (emb)',       'mlp_emb',       'mlp',  'emb'),
    ('MLP (all)',       'mlp_all',       'mlp',  ['structure', 'emb']),
    ('LightGBM (all)',  'lightgbm_all',  'lgbm', 'all'),
    ('XGBoost (all)',   'xgboost_all',   'xgb',  'all'),
]

# GNN models: (display name, checkpoint stem, feature_groups for pairwise features)
GNN_REGISTRY = [
    ('GNN+MLP (gnn+emb)',         'gnn_mlp_gnn_emb',         'emb'),
    ('GNN+MLP (gnn+struct+emb)',  'gnn_mlp_gnn_struct_emb',  ['structure', 'emb']),
]

def _fg_key(fg):
    return json.dumps(fg, sort_keys=True) if isinstance(fg, list) else fg

# Collect all unique feature groups we'll need (MLP/boosting + GNN pairwise features)
needed_fgs = {}
for _, _, _, fg in MODEL_REGISTRY:
    needed_fgs[_fg_key(fg)] = fg
for _, _, fg in GNN_REGISTRY:
    needed_fgs[_fg_key(fg)] = fg

# Fit scalers via load_datasets and keep the full DataSplit (needed later for GNNData).
split_by_fg  = {}
scaler_by_fg = {}
for key, fg in needed_fgs.items():
    ds = load_datasets(str(TRAIN_CSV), str(VAL_CSV), str(RANKING_CSV), fg)
    split_by_fg[key]  = ds
    scaler_by_fg[key] = ds.scaler
    print(f'feature group {fg} -> n_features = {ds.train_ds.X.shape[1]}')

feature group structure -> n_features = 41
feature group emb -> n_features = 2
feature group ['structure', 'emb'] -> n_features = 43
feature group all -> n_features = 43


## Load saved models (MLP / boosting / GNN)

In [6]:
from src.link_prediction.models.mlp import MLPTrainer
from src.link_prediction.models.boosting_models import LightGBMModel, XGBoostModel

# loaded_models: name -> dict with keys:
#   'predict'    : callable
#       - for non-GNN models: predict(X)              -> scores
#       - for GNN models    : predict(df, X)          -> scores
#   'kind'       : 'mlp' | 'lgbm' | 'xgb' | 'gnn'
#   'fg'         : feature group (for pairwise features)
loaded_models = {}

for name, stem, kind, fg in MODEL_REGISTRY:
    path = MODEL_DIR / (f'{stem}.pt' if kind == 'mlp' else f'{stem}.pkl')
    print(f'LOAD  {name} - {path.name}')
    if not path.is_file():
        print(f'   SKIP (missing file)')
        continue
    try:
        if kind == 'mlp':
            obj = MLPTrainer.load(str(path))
            predict = lambda X, _o=obj: _o.predict(X)
        elif kind == 'lgbm':
            obj = LightGBMModel.load(str(path))
            predict = lambda X, _o=obj: _o.predict(X)
        elif kind == 'xgb':
            obj = XGBoostModel.load(str(path))
            predict = lambda X, _o=obj: _o.predict(X)
        loaded_models[name] = {'predict': predict, 'kind': kind, 'fg': fg}
        print(f'   OK ({name})')
    except Exception as e:
        print(f'   FAILED to load {name}: {type(e).__name__}: {e}')

# --- GNN models ---
if HAS_TORCH_GEOMETRIC and GRAPH_FILE.is_file():
    from src.link_prediction.models.graph_models import GNNTrainer

    # Cache GNNData per feature group so we don't rebuild the graph twice.
    gnn_data_by_fg = {}

    for name, stem, fg in GNN_REGISTRY:
        path = MODEL_DIR / f'{stem}.pt'
        print(f'LOAD  {name} - {path.name}')
        if not path.is_file():
            print(f'   SKIP (missing file)')
            continue
        try:
            key = _fg_key(fg)
            if key not in gnn_data_by_fg:
                gnn_data_by_fg[key] = GNNData(split_by_fg[key], str(GRAPH_FILE))
            gnn_data = gnn_data_by_fg[key]

            trainer = GNNTrainer.load(
                str(path), gnn_data.graph, gnn_data.node_features
            )
            c2i = gnn_data.concept_to_idx

            def _make_gnn_fn(t, c):
                def fn(df, X):
                    vp = np.array(
                        [[c.get(a, 0), c.get(b, 0)]
                         for a, b in zip(df['concept_a'], df['concept_b'])],
                        dtype=np.int64,
                    )
                    return t.predict(vp, X)
                return fn

            loaded_models[name] = {
                'predict': _make_gnn_fn(trainer, c2i),
                'kind': 'gnn',
                'fg': fg,
            }
            print(f'   OK ({name})')
        except Exception as e:
            print(f'   FAILED to load {name}: {type(e).__name__}: {e}')
else:
    if not HAS_TORCH_GEOMETRIC:
        print('Skipping GNN models: torch_geometric not installed.')
    elif not GRAPH_FILE.is_file():
        print(f'Skipping GNN models: graph file not found at {GRAPH_FILE}')

print('\nLoaded models:', list(loaded_models.keys()))

LOAD  MLP (structure) - mlp_structure.pt
   OK (MLP (structure))
LOAD  MLP (emb) - mlp_emb.pt
   OK (MLP (emb))
LOAD  MLP (all) - mlp_all.pt
   OK (MLP (all))
LOAD  LightGBM (all) - lightgbm_all.pkl
   OK (LightGBM (all))
LOAD  XGBoost (all) - xgboost_all.pkl
   OK (XGBoost (all))
LOAD  GNN+MLP (gnn+emb) - gnn_mlp_gnn_emb.pt
   OK (GNN+MLP (gnn+emb))
LOAD  GNN+MLP (gnn+struct+emb) - gnn_mlp_gnn_struct_emb.pt
   OK (GNN+MLP (gnn+struct+emb))

Loaded models: ['MLP (structure)', 'MLP (emb)', 'MLP (all)', 'LightGBM (all)', 'XGBoost (all)', 'GNN+MLP (gnn+emb)', 'GNN+MLP (gnn+struct+emb)']


## Prepare ranking test set

In [7]:
df_rank = pd.read_csv(RANKING_CSV)
print('ranking_test.csv:', df_rank.shape)

y_rank = df_rank['label'].values.astype(np.int32)
print('positives:', int(y_rank.sum()), '/', len(y_rank))

# Precompute scaled feature matrices, one per unique feature group.
groups = classify_features(df_rank.columns.tolist())

X_by_fg = {}
for key, fg in needed_fgs.items():
    cols = _select_cols(groups, fg)
    if not cols:
        print(f'WARN: no columns for feature group {fg}')
        continue
    X = np.nan_to_num(df_rank[cols].values.astype(np.float32))
    X = scaler_by_fg[key].transform(X).astype(np.float32)
    X_by_fg[key] = X
    print(f'feature group {fg} -> X shape {X.shape}')

ranking_test.csv: (291739, 49)
positives: 1552 / 291739
feature group structure -> X shape (291739, 41)
feature group emb -> X shape (291739, 2)
feature group ['structure', 'emb'] -> X shape (291739, 43)
feature group all -> X shape (291739, 43)


## Evaluate models

In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
rank_results = {}   # name -> metrics dict

for name, info in loaded_models.items():
    predict = info['predict']
    kind    = info['kind']
    fg      = info['fg']
    key     = _fg_key(fg)

    if key not in X_by_fg:
        print(f'SKIP {name}: no X for feature group {fg}')
        continue
    X = X_by_fg[key]

    try:
        if kind == 'gnn':
            scores = predict(df_rank, X)
        else:
            scores = predict(X)
    except Exception as e:
        print(f'SKIP {name}: predict failed ({type(e).__name__}: {e})')
        continue

    scores = np.asarray(scores).ravel()
    if scores.shape[0] != y_rank.shape[0]:
        print(f'SKIP {name}: bad output shape {scores.shape}')
        continue

    metrics = compute_ranking_metrics(y_rank, scores, K_VALUES)
    rank_results[name] = metrics
    print(f'OK {name}')

OK MLP (structure)
OK MLP (emb)
OK MLP (all)
OK LightGBM (all)
OK XGBoost (all)
OK GNN+MLP (gnn+emb)
OK GNN+MLP (gnn+struct+emb)


In [10]:
wanted_cols = ['model', 'roc_auc', 'mrr',
               'ndcg_at_5', 'ndcg_at_10', 'ndcg_at_50', 'ndcg_at_100']

rows = []
for name, m in rank_results.items():
    row = {'model': name}
    for c in wanted_cols[1:]:
        row[c] = m.get(c)
    rows.append(row)

summary_df = pd.DataFrame(rows, columns=wanted_cols)
summary_df

,model,roc_auc,mrr,ndcg_at_5,ndcg_at_10,ndcg_at_50,ndcg_at_100
0,MLP (structure),0.887623,0.100000,0.000000,0.063621,0.183577,0.210412
1,MLP (emb),0.850443,0.023256,0.000000,0.000000,0.014202,0.024241
2,MLP (all),0.907381,0.250000,0.146068,0.094788,0.297210,0.296935
3,LightGBM (all),0.862720,0.016667,0.000000,0.000000,0.000000,0.008053
4,XGBoost (all),0.890436,0.500000,0.213986,0.202483,0.088240,0.054354
5,GNN+MLP (gnn+emb),0.836513,0.038462,0.000000,0.000000,0.032434,0.027646
6,GNN+MLP (gnn+struct+emb),0.859918,0.111111,0.000000,0.129875,0.287491,0.261266
